In [24]:
import ollama
import json

In [25]:
with open("data/grids.json", "r") as f:
    grids = json.load(f)

In [26]:
MODEL = "mistral"

In [27]:
SYSTEM_PROMPT = """
You are a helpful assistant that helps a robot navigate a grid world. 

Environment:
    A robot is placed in a 5x5 grid world:
        - The top left corner of the grid is (0, 0) and the bottom right corner is (4, 4).
        - The robot can only move to open cells.
        - The robot cannot move into blocked cells.
        - The robot can move to four directions: North, South, East, and West.
        - North moves the robot up by one cell.
        - South moves the robot down by one cell.
        - East moves the robot right by one cell.
        - West moves the robot left by one cell.
        - The robot cannot move outside the boundaries of the grid.

Task:
    - Provide a path for the robot to reach the goal from the starting position while avoiding the blocked cells and staying within the grid boundaries.

Output:
    - Mention the position of the robot after each step.
    - Finally, after reaching the goal, provide an array of directions (North, South, East, West) that the robot should take to reach the goal from the starting position.

#####################

Example 1:

Input:
Starting position: [0, 0]
Goal position: [4, 4]
Obstacles: [[0, 1], [0, 3], [1, 2], [1, 3]]

Output:
    - Position: [0, 0] -> [1, 0] (South)
    - Position: [1, 0] -> [1, 1] (East)
    - Position: [1, 1] -> [2, 1] (South)
    - Position: [2, 1] -> [2, 2] (East)
    - Position: [2, 2] -> [3, 2] (South)
    - Position: [3, 2] -> [3, 3] (East)
    - Position: [3, 3] -> [4, 3] (South)
    - Position: [4, 3] -> [4, 4] (East)

Array: ['South', 'East', 'South', 'East', 'East', 'East', 'South', 'South']

#####################

Example 2:

Input:
Grid:
. . . . .
. G B . .
. . B B S
. . B B .
. . . B .
Starting position: [2, 4]
Goal position: [1, 1]
Obstacles: [[1, 2], [2, 2], [2, 3], [3, 2], [3, 3], [4, 3]]

Output:
    - Position: [2, 4] -> [1, 4] (North)
    - Position: [1, 4] -> [0, 4] (North)
    - Position: [0, 4] -> [0, 3] (West)
    - Position: [0, 3] -> [0, 2] (West)
    - Position: [0, 2] -> [0, 1] (West)
    - Position: [0, 1] -> [1, 1] (South)

Array: ['North', 'North', 'West', 'West', 'West', 'South']

"""

In [28]:
def ask_slm(prompt):
    print("Asking SLM...")
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )

    return response["message"]["content"]

In [29]:
for grid_info in grids[:5]:
    grid = grid_info["grid"]
    start = grid_info["start"]
    goal = grid_info["goal"]
    obstacles = grid_info["obstacles"]

    print(f"Grid:\n{'\n'.join(' '.join(row) for row in grid)}")
    prompt = f"Starting position: {start}\nGoal position: {goal}\nObstacles: {obstacles}"
    print(f"Prompt:\n{prompt}\n")

    response = ask_slm(prompt)
    print(f"Response:\n{response}\n")

Grid:
S . . . B
B . . . .
. G . . .
. . . . B
. . . B .
Prompt:
Starting position: [0, 0]
Goal position: [2, 1]
Obstacles: [[1, 0], [0, 4], [3, 4], [4, 3]]

Asking SLM...
Response:
 In this example, the robot starts at position [0, 0] and needs to reach position [2, 1] avoiding the blocked cells. Here is the path the robot should take:

- Position: [0, 0] -> [0, 1] (North)
- Position: [0, 1] -> [1, 1] (East)
- Position: [1, 1] -> [1, 2] (South)
- Position: [1, 2] -> [2, 2] (East)
- Position: [2, 2] -> [2, 3] (North)
- Position: [2, 3] -> [3, 3] (East)
- Position: [3, 3] -> [3, 4] (South)
- Position: [3, 4] -> [2, 4] (West)
- Position: [2, 4] -> [2, 3] (West)
- Position: [2, 3] -> [2, 2] (West)
- Position: [2, 2] -> [2, 1] (West)

Array: ['North', 'East', 'South', 'East', 'North', 'East', 'South', 'West', 'West', 'West', 'West']

Grid:
. . B . S
. . B B .
. . . . .
G . . B .
. . . . .
Prompt:
Starting position: [0, 4]
Goal position: [3, 0]
Obstacles: [[0, 2], [1, 2], [1, 3], [3, 3]]

As

In [14]:
user_prompt = f"Grid:\n{'\n'.join(' '.join(row) for row in grid_info['grid'])}\nStarting position: {grid_info['start']}\nGoal position: {grid_info['goal']}."

In [15]:
print(user_prompt)

Grid:
. B B . S
. . B . .
. . . . G
. . . . .
B . . . B
Starting position: [0, 4]
Goal position: [2, 4].


In [16]:
ask_slm(user_prompt)

Asking SLM...


" ['North', 'North', 'North', 'North']"

In [17]:
SYSTEM_PROMPT = """
You are a grid world navigation engine. 

A robot is on a 5x5 coordinate plane where (0,0) is top-left and (4,4) is bottom-right.
Movement rules:
- North: (x-1, y)
- South: (x+1, y)
- East:  (x, y+1)
- West:  (x, y-1)

CRITICAL: Do not output any Python code, BFS algorithms, or introductory conversational text. You must output ONLY a valid JSON array of strings representing the path directions.

Input Format:
Starting position: [x, y]
Goal position: [x, y]
Blocked cells: [[x, y], [x, y]...]
Open cells: All cells not blocked.
"""